# WDPA integrated alerts area: per-tile point coverage

Integrated disturbance alert area for every protected area, using the point-coverage
method from `wdpa_tcl_area_coverage.ipynb` (de-duplicated geometries, worker-side geometry
loading, bounded-memory combine and write).

**Inputs** (same as the analytics API's live path and the admin pipeline): the integrated
alerts zarr at the `latest` version (10 m, `alert_date` = days since 2014-12-31,
`confidence` 2/3/4), 10 m pixel area (m²), and SBTN natural lands (30 m, resampled
nearest-neighbour onto the 10 m grid).

**Output**: one row per site x natural lands class x alert date x confidence, the same
schema as the admin pipeline's table, so every filter the API offers (date range,
`land_filter`) can be applied at query time. The optional `START_DATE`, `END_DATE` and
`LAND_FILTER` constants apply the same filters before computing instead.

**Per tile** (1 x 1 degree at 10 m): load the four layers once, give each alert pixel a
combined (class, date, confidence) code, then for each polygon test only the alert pixels
in its bbox window (`shapely.contains_xy`) and sum pixel area per code present
(`np.unique` + `bincount`).

In [ ]:
import dask
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import shapely
from shapely import STRtree

In [ ]:
import logging
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [ ]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
import fsspec

ANALYTICS_BUCKET = "lcl-analytics"
S3_OPTS = {"requester_pays": True}

# Alerts version: the same `latest` marker the analytics API resolves; pin a version
# string here instead to reproduce a run.
ALERTS_VERSION = fsspec.open(
    f"s3://{ANALYTICS_BUCKET}/zonal-statistics/integrated-alerts/latest", "rt", **S3_OPTS
).open().read().strip()
alerts_zarr_uri = f"s3://{ANALYTICS_BUCKET}/zarr/gfw_integrated_dist_alerts/{ALERTS_VERSION}/date_conf.zarr"
# Same inputs as the API's live path and the admin pipeline
pixel_area_10m_zarr_uri = "s3://gfw-data-lake/umd_area_2013/v1.10/raster/epsg-4326/zarr/area_m_10m_f32"  # m²
natural_lands_zarr_uri = f"s3://{ANALYTICS_BUCKET}/zarr/sbtn-natural-lands/sbtn_natural_lands_all_classes.zarr"

# Alert encoding (as in the API): alert_date = days since 2014-12-31, confidence 2/3/4.
ALERT_EPOCH = pd.Timestamp("2014-12-31")
N_DATES = 5000                      # alert_date codes 0..4999, the API's expected groups
CONFIDENCE = {2: "low", 3: "high", 4: "highest"}
N_CLASSES = 22                      # raw SBTN natural lands classes 0..21 (0 = no data)

# Group layers: name -> (min value, number of values). They match the admin pipeline's
# table, so every API filter can be served from the output (land_filter via
# natural_lands_class, dates via alert_date). A pixel counts only if every layer is in
# range; adding a layer is one entry here plus its variable in load_tile.
GROUP_LAYERS = {
    "natural_lands_class": (0, N_CLASSES),
    "alert_date": (0, N_DATES),
    "alert_confidence": (min(CONFIDENCE), len(CONFIDENCE)),
}
CONTEXT_NAMES = list(GROUP_LAYERS)
VALUE_LAYERS = ["area_ha"]

# Optional filters, as in the API request (all None = keep everything, the default).
# Filtering here removes candidate pixels before the point test, so runs get faster.
START_DATE = None                   # e.g. "2024-01-01"
END_DATE = None                     # e.g. "2024-12-31"
LAND_FILTER = None                  # None, "natural_lands" or "natural_forests"
# Keep in sync with LAND_FILTER_CLASSES in api/app/analysis/common/analysis.py
LAND_FILTER_CLASSES = {"natural_lands": list(range(2, 12)), "natural_forests": [2, 5, 8, 9]}

# 10000 px = one zarr chunk = 1 x 1 degree at 10 m. 20000 (2 x 2 chunks) gives 4x fewer
# tasks at ~4 GB peak per task.
TILE_PX = 10000

# Rows of a polygon's window tested at once; bounds the point arrays for large polygons
# over dense alerts (1000 rows x 10000 columns is at most 10M points).
ROW_BLOCK = 1000

WDPA_GEOPARQUET_URI = "s3://lcl-analytics/vectors/wdpa.parquet"

# Restrict to protected areas in one country (ISO3), or None for global.
TEST_ISO3 = None

RUN_NAME = TEST_ISO3 or "global"
RESULT_URI = (f"s3://{ANALYTICS_BUCKET}/zonal-statistics/integrated-alerts/test/"
              f"wdpa-integrated-alerts-{RUN_NAME}-{ALERTS_VERSION}.parquet")
print(f"alerts {ALERTS_VERSION} -> {RESULT_URI}")

In [ ]:
# ---------------------------------------------------------------------------
# Load WDPA areas
# ---------------------------------------------------------------------------
import pyarrow.compute as pc

# Filter and prune columns while reading: the file is ~5 GB, and loading it whole
# leaves the kernel little headroom for the rest of the run.
WDPA_COLUMNS = ["SITE_ID", "SITE_PID", "NAME", "DESIG_ENG", "ISO3", "REALM", "GIS_AREA", "geometry"]
row_filter = (pc.field("REALM") != "Marine") | pc.field("REALM").is_null()
if TEST_ISO3:
    row_filter &= pc.match_substring(pc.field("ISO3"), TEST_ISO3)
wdpa_gdf = gpd.read_parquet(WDPA_GEOPARQUET_URI, columns=WDPA_COLUMNS, filters=row_filter)

if TEST_ISO3:
    # match_substring is a coarse prefilter; keep only exact codes in "BRA;PRY"-style lists
    in_country = wdpa_gdf["ISO3"].str.split(";").apply(lambda codes: TEST_ISO3 in codes)
    wdpa_gdf = wdpa_gdf[in_country]
wdpa_gdf = wdpa_gdf.reset_index(drop=True)

# Repair invalid geometries once, up front, so the STRtree queries in tiling and
# coloring never hit a GEOS topology error. "structure" + keep_collapsed=False
# keeps only polygonal output: the default method can return stray LineStrings,
# which rasterio would burn into the label raster.
invalid = ~shapely.is_valid(wdpa_gdf.geometry.values)
wdpa_gdf.loc[invalid, "geometry"] = shapely.make_valid(
    wdpa_gdf.geometry.values[invalid], method="structure", keep_collapsed=False
)
wdpa_gdf = wdpa_gdf[~wdpa_gdf.geometry.is_empty].reset_index(drop=True)

site_geoms = wdpa_gdf.geometry.values
site_ids = wdpa_gdf["SITE_ID"].values


def dedupe_geometries(geoms, chunk=10_000):
    """Unique geometries, plus the index of each input row's unique geometry.

    Hashes WKB a chunk at a time so only 8 bytes per row are held, instead of
    a full WKB copy of every geometry.
    """
    hashes = np.empty(len(geoms), dtype=np.uint64)
    for i in range(0, len(geoms), chunk):
        hashes[i:i + chunk] = pd.util.hash_array(shapely.to_wkb(geoms[i:i + chunk]))
    codes, _ = pd.factorize(hashes)
    first = pd.Series(np.arange(len(geoms))).groupby(codes).first().to_numpy()
    # A 64-bit hash collision would merge different geometries; check, don't assume.
    assert shapely.equals_identical(geoms, geoms[first][codes]).all(), "WKB hash collision"
    return geoms[first], codes


# Tiles only see unique geometries (GEOM_ID); results are copied back to every site
# sharing one in the combine step. Byte-identical geometries are common (SAC/SPA
# pairs) and sometimes stacked thousands deep (Finland), and every stacked copy costs
# a coloring pass or superlinear union_all time. A site with several identical rows
# counts the geometry once.
geometries, row_geom_id = dedupe_geometries(site_geoms)
ids = np.arange(len(geometries))
geom_to_site = pd.DataFrame({"GEOM_ID": row_geom_id, "SITE_ID": site_ids}).drop_duplicates()

print(f"Loaded {len(site_geoms)} WDPA areas ({invalid.sum()} repaired), "
      f"{len(geometries)} unique geometries" + (f" for {TEST_ISO3}" if TEST_ISO3 else ""))

In [ ]:
# ---------------------------------------------------------------------------
# Tile loading on the 10 m alerts grid (same alignment as the API and pipeline)
# ---------------------------------------------------------------------------
def _open_zarr(name):
    uri = {"alerts": alerts_zarr_uri, "pixel_area": pixel_area_10m_zarr_uri,
           "natural_lands": natural_lands_zarr_uri}[name]
    ds = xr.open_zarr(uri, storage_options=S3_OPTS)
    return ds.squeeze("band", drop=True) if "band" in ds.dims else ds


def _zarr(name):
    """Open each zarr once per worker process: at 10 m the coordinate arrays alone are
    tens of MB, too much to re-read for every tile.

    The cache lives on the worker object, never in a notebook global: cloudpickle ships
    a notebook function's globals with it, so a client-side cache holding an opened
    zarr made every task ~50 MB and overwhelmed the scheduler.
    """
    try:
        from distributed import get_worker
        cache = get_worker().__dict__.setdefault("_wdpa_alert_zarrs", {})
    except ValueError:  # not on a worker (client, or local tests): open, don't cache
        return _open_zarr(name)
    if name not in cache:
        cache[name] = _open_zarr(name)
    return cache[name]


def alerts_grid():
    """The lazy alerts array whose chunks define the tiles."""
    return _zarr("alerts")["confidence"]


def load_tile(ys, xs, bbox):
    """Alerts, pixel area (ha) and natural lands class for one tile, on the alerts grid."""
    alerts = _zarr("alerts").isel(y=ys, x=xs)
    # Pixel area shares the alerts grid (a 100000-row offset), so match within 1e-5 deg.
    area = (_zarr("pixel_area")["band_data"]
            .sel(x=slice(bbox[0], bbox[2]), y=slice(bbox[3], bbox[1]))
            .reindex_like(alerts, method="nearest", tolerance=1e-5))
    # 30 m natural lands: each 10 m pixel takes the class of the 30 m pixel its center
    # falls in (tolerance half a 30 m pixel, outside the layer -> 0), as in the pipeline.
    nl = _zarr("natural_lands")["band_data"]
    res = abs(float(nl.x[1] - nl.x[0]))
    nl = (nl.sel(x=slice(bbox[0] - res, bbox[2] + res), y=slice(bbox[3] + res, bbox[1] - res))
          .reindex_like(alerts, method="nearest", tolerance=res / 2, fill_value=0))
    return xr.Dataset({
        "alert_date": alerts["alert_date"], "alert_confidence": alerts["confidence"],
        "area_m2": area, "natural_lands_class": nl,
    }).load()


ds_alerts = alerts_grid()
print("alerts grid (lazy)", dict(ds_alerts.sizes))

In [ ]:
# ---------------------------------------------------------------------------
# Tiling: enumerate windows; workers load and clip their own geometries
# ---------------------------------------------------------------------------
import fsspec
import pyarrow as pa
import pyarrow.parquet as pq


def _chunk_edges(chunks, target):
    """Coalesce dask chunk boundaries into tile edges of at least `target` pixels."""
    bounds = np.concatenate([[0], np.cumsum(chunks)])
    edges = [0]
    for b in bounds[1:]:
        if b - edges[-1] >= target or b == bounds[-1]:
            edges.append(int(b))
    return edges


def tile_windows(da, tile_px):
    """Yield (y_slice, x_slice, bbox) for tiles snapped to chunk boundaries.
    bbox is in pixel-edge coordinates (coords are assumed to be pixel centers)."""
    x, y = da.x.values, da.y.values
    dx, dy = abs(float(x[1] - x[0])), abs(float(y[1] - y[0]))
    ye = _chunk_edges(da.chunks[da.get_axis_num("y")], tile_px)
    xe = _chunk_edges(da.chunks[da.get_axis_num("x")], tile_px)
    for y0, y1 in zip(ye[:-1], ye[1:]):
        for x0, x1 in zip(xe[:-1], xe[1:]):
            bbox = (x[x0] - dx / 2, y[y1 - 1] - dy / 2, x[x1 - 1] + dx / 2, y[y0] + dy / 2)
            yield slice(y0, y1), slice(x0, x1), bbox


def _storage_options(uri):
    return S3_OPTS if uri.startswith("s3://") else None


def write_sorted_geometries(geoms, uri, row_group_size=1_000, chunk=20_000):
    """Write unique geometries as WKB with bbox columns, Hilbert-sorted, so a worker's
    bbox filter only reads the few row groups near its tile.

    Geometries wider than a tile go last, in their own row groups, so their wide
    bboxes don't make every row group look relevant to every tile. Written a chunk at
    a time to keep client memory bounded.
    """
    b = shapely.bounds(geoms)
    wide = ((b[:, 2] - b[:, 0]) > 2.5) | ((b[:, 3] - b[:, 1]) > 2.5)
    hilbert = gpd.GeoSeries(geoms).hilbert_distance(total_bounds=(-180, -90, 180, 90)).to_numpy()
    order = np.lexsort((hilbert, wide))
    schema = pa.schema([("GEOM_ID", pa.int64()), ("xmin", pa.float64()), ("ymin", pa.float64()),
                        ("xmax", pa.float64()), ("ymax", pa.float64()), ("wkb", pa.binary())])
    with fsspec.open(uri, "wb", **(_storage_options(uri) or {})) as f, pq.ParquetWriter(f, schema) as w:
        for i in range(0, len(order), chunk):
            idx = order[i:i + chunk]
            w.write_table(pa.table({
                "GEOM_ID": idx, "xmin": b[idx, 0], "ymin": b[idx, 1], "xmax": b[idx, 2], "ymax": b[idx, 3],
                "wkb": shapely.to_wkb(geoms[idx]),
            }, schema=schema), row_group_size=row_group_size)


def build_tile_tasks(da, tree, tile_px):
    """(ys, xs, bbox) for each tile that intersects at least one geometry.

    Tasks carry no geometry, so nothing heavy is serialized on the client: workers
    load their own with load_tile_geoms.
    """
    tasks, n_total = [], 0
    for ys, xs, bbox in tile_windows(da, tile_px):
        n_total += 1
        if len(tree.query(shapely.box(*bbox), predicate="intersects")):
            tasks.append((ys, xs, bbox))
    print(f"{len(tasks)} of {n_total} tiles intersect protected areas")
    return tasks


def clip_to_tile(geoms, geom_ids, bbox):
    """Clip to the tile; an empty clip also drops geometries that only touch it."""
    clipped = shapely.clip_by_rect(geoms, *bbox)
    keep = ~shapely.is_empty(clipped)
    return clipped[keep], geom_ids[keep]


def load_tile_geoms(bbox):
    """Worker side: unique geometries touching a tile, read from GEOMS_URI and clipped."""
    x0, y0, x1, y1 = bbox
    df = pd.read_parquet(
        GEOMS_URI, columns=["GEOM_ID", "wkb"], storage_options=_storage_options(GEOMS_URI),
        filters=[("xmin", "<=", x1), ("xmax", ">=", x0), ("ymin", "<=", y1), ("ymax", ">=", y0)],
    )
    return clip_to_tile(shapely.from_wkb(df["wkb"].to_numpy()), df["GEOM_ID"].to_numpy(), bbox)


def tile_geoms_local(bbox):
    """Client side equivalent of load_tile_geoms, for diagnostics; one tile at a time."""
    hits = geom_tree.query(shapely.box(*bbox), predicate="intersects")
    return clip_to_tile(geometries[hits], ids[hits], bbox)


def run_tile(task):
    ys, xs, bbox = task
    geoms, geom_ids = load_tile_geoms(bbox)
    if len(geoms) == 0:
        return pd.DataFrame()
    # Load the tile's zarr chunks inside this task. Without this, .load() on a worker
    # submits a nested graph back to the cluster and the tile task blocks on it.
    with dask.config.set(scheduler="synchronous"):
        return process_tile((ys, xs, bbox, geoms, geom_ids))

In [ ]:
# ---------------------------------------------------------------------------
# Per-tile worker: point coverage on alert pixels, sparse grouping per polygon
# ---------------------------------------------------------------------------
GROUP_OFFSETS = [low for low, _ in GROUP_LAYERS.values()]
GROUP_DIMS = [n for _, n in GROUP_LAYERS.values()]
assert np.prod(GROUP_DIMS, dtype=object) < 2**63, "group layers too large to pack into int64"


def _day(date_str):
    return (pd.Timestamp(date_str) - ALERT_EPOCH).days


def _candidates(tile):
    """True where a pixel counts: every group layer in range (so an alert is present,
    since confidence is 0 without one) and the optional filters pass."""
    candidate = np.ones((tile.sizes["y"], tile.sizes["x"]), dtype=bool)
    for name, (low, n) in GROUP_LAYERS.items():
        values = tile[name].values
        candidate &= (values >= low) & (values < low + n)
    if START_DATE is not None:
        candidate &= tile["alert_date"].values >= _day(START_DATE)
    if END_DATE is not None:
        candidate &= tile["alert_date"].values <= _day(END_DATE)
    if LAND_FILTER is not None:
        candidate &= np.isin(tile["natural_lands_class"].values, LAND_FILTER_CLASSES[LAND_FILTER])
    return candidate


def _pixel_windows(geoms, bbox, height, width):
    """Pixel window of each geometry's bounding box, clipped to the tile: arrays of
    row_start, row_stop, col_start, col_stop (stop exclusive, like slices)."""
    tile_west, _, _, tile_north = bbox
    pixel_width = (bbox[2] - bbox[0]) / width
    pixel_height = (bbox[3] - bbox[1]) / height
    west, south, east, north = shapely.bounds(geoms).T
    col_start = np.clip(np.floor((west - tile_west) / pixel_width), 0, width).astype(int)
    col_stop = np.clip(np.ceil((east - tile_west) / pixel_width), 0, width).astype(int)
    row_start = np.clip(np.floor((tile_north - north) / pixel_height), 0, height).astype(int)
    row_stop = np.clip(np.ceil((tile_north - south) / pixel_height), 0, height).astype(int)
    return row_start, row_stop, col_start, col_stop


def process_tile(task):
    ys, xs, bbox, geoms, geom_ids = task
    tile = load_tile(ys, xs, bbox)
    height, width = tile.sizes["y"], tile.sizes["x"]
    candidate = _candidates(tile)
    if not candidate.any():
        return pd.DataFrame()
    layers = [tile[name].values for name in GROUP_LAYERS]
    area_m2 = tile["area_m2"].values

    row_start, row_stop, col_start, col_stop = _pixel_windows(geoms, bbox, height, width)
    x_centers, y_centers = tile.x.values, tile.y.values  # pixel centers, as stored in the zarr
    geom_id_parts, group_parts, area_parts = [], [], []
    for i, geom in enumerate(geoms):
        if row_start[i] >= row_stop[i] or col_start[i] >= col_stop[i]:
            continue
        shapely.prepare(geom)
        window_cols = slice(col_start[i], col_stop[i])
        for block_start in range(row_start[i], row_stop[i], ROW_BLOCK):
            block_stop = min(block_start + ROW_BLOCK, row_stop[i])
            window = (slice(block_start, block_stop), window_cols)
            # Alert pixels in this block of the window (row, col within the block); alerts
            # are sparse, so only these are tested.
            alert_rows, alert_cols = np.nonzero(candidate[window])
            if len(alert_rows) == 0:
                continue
            inside = shapely.contains_xy(geom, x_centers[col_start[i] + alert_cols],
                                         y_centers[block_start + alert_rows])
            if not inside.any():
                continue
            alert_rows, alert_cols = alert_rows[inside], alert_cols[inside]
            # One integer per pixel for its combination of group layers (np.ravel_multi_index
            # packs and bounds-checks them), so a single unique + bincount sums area per group.
            # The group space is too big for a dense bincount per polygon, so sum only over
            # the groups present in this block.
            pixel_group = np.ravel_multi_index(
                [layer[window][alert_rows, alert_cols].astype(np.int64) - low
                 for layer, low in zip(layers, GROUP_OFFSETS)], GROUP_DIMS)
            groups_present, group_index = np.unique(pixel_group, return_inverse=True)
            pixel_area_ha = np.nan_to_num(area_m2[window][alert_rows, alert_cols].astype(np.float64)) / 10_000
            geom_id_parts.append(np.full(len(groups_present), geom_ids[i]))
            group_parts.append(groups_present)
            area_parts.append(np.bincount(group_index, weights=pixel_area_ha, minlength=len(groups_present)))

    if not geom_id_parts:
        return pd.DataFrame()
    df = pd.DataFrame({"GEOM_ID": np.concatenate(geom_id_parts), "group": np.concatenate(group_parts),
                       "area_ha": np.concatenate(area_parts)})
    df = df.groupby(["GEOM_ID", "group"], as_index=False)["area_ha"].sum()  # merge row blocks
    unpacked = np.unravel_index(df.pop("group").to_numpy(), GROUP_DIMS)
    for name, low, values in zip(GROUP_LAYERS, GROUP_OFFSETS, unpacked):
        df[name] = values + low
    return df[df["area_ha"] > 0]

In [ ]:
# ---------------------------------------------------------------------------
# Build tasks: write the geometries workers read, then the (geometry-free) tile tasks
# ---------------------------------------------------------------------------
GEOMS_URI = f"s3://{ANALYTICS_BUCKET}/zonal-statistics/integrated-alerts/test/wdpa-unique-geoms-{RUN_NAME}.parquet"
write_sorted_geometries(geometries, GEOMS_URI)
geom_tree = STRtree(geometries)
tasks = build_tile_tasks(ds_alerts, geom_tree, TILE_PX)

In [ ]:
import coiled

cluster = coiled.Cluster(
    name="wdpa_integrated_alerts_coverage",
    region="us-east-1",
    n_workers=10,
    tags={"project": "gnw_zonal_stat"},
    scheduler_vm_types=["r7i.xlarge"],
    worker_vm_types=["r7i.2xlarge"],
    compute_purchase_option="on-demand",
    no_client_timeout="10 minutes",
    worker_options={"nthreads": 4},
)

dask_client = cluster.get_client()
cluster.adapt(minimum=10, maximum=40)

In [ ]:
%%time
# ---------------------------------------------------------------------------
# Run: one task per tile, gather partials as they finish
# ---------------------------------------------------------------------------
from dask.distributed import as_completed

# Tasks are just tile windows; each worker reads and clips its own geometries.
# pure=False: fresh keys each run, so a re-run can't inherit cancelled futures from
# an earlier attempt that are still referenced in the kernel.
futures = dask_client.map(run_tile, tasks, pure=False)
# Results come back in batches (everything finished so far, one request each) rather
# than one round trip per tile; with thousands of tiles the per-tile .result() calls
# fell behind the cluster. raise_errors=False hands failed tiles back in the batch.
parts, failed, done = [], [], 0
for batch in as_completed(futures, with_results=True, raise_errors=False).batches():
    for fut, res in batch:
        if fut.status == "error":
            failed.append((fut.key, repr(fut.exception())))
        elif not res.empty:
            parts.append(res)
    done += len(batch)
    print(f"\r{done:,}/{len(futures):,} tiles collected, {len(failed)} failed", end="", flush=True)

print(f"\n{len(parts)} non-empty tiles, {len(failed)} failed")
if failed:
    print(failed[:5])

In [ ]:
# ---------------------------------------------------------------------------
# Combine partials per unique geometry (bounded memory on the client)
# ---------------------------------------------------------------------------
import gc

assert not failed, f"{len(failed)} tiles failed; fix or rerun them before combining"

for _name in ("wdpa_gdf", "site_geoms", "geometries", "geom_tree", "row_geom_id"):
    globals().pop(_name, None)
gc.collect()

KEY_DTYPES = {"GEOM_ID": "int32", "natural_lands_class": "int8", "alert_date": "int16", "alert_confidence": "int8"}
raw = pd.concat([p.astype(KEY_DTYPES) for p in parts if not p.empty], ignore_index=True)
del parts
gc.collect()

# One row per geometry x group: partials of geometries spanning several tiles are summed.
by_geom = raw.groupby(["GEOM_ID", *CONTEXT_NAMES], as_index=False)[VALUE_LAYERS].sum()
n_raw = len(raw)
del raw
gc.collect()

geom_to_site = geom_to_site.astype({"GEOM_ID": "int32"})
rows_per_geom = by_geom.groupby("GEOM_ID").size()
expanded = int(geom_to_site["GEOM_ID"].map(rows_per_geom).fillna(0).sum())
print(f"partial rows: {n_raw:,} -> per-geometry rows: {len(by_geom):,}; "
      f"copying to sites gives {expanded:,} rows before the per-site sum")

In [ ]:
# ---------------------------------------------------------------------------
# Postprocessing: expand to sites, label, and write, a chunk of sites at a time
# ---------------------------------------------------------------------------
# Same schema as the admin pipeline's table (aoi_id, natural_lands_class, alert_date,
# alert_confidence, area_ha), with aoi_type = protected_area.
assert not failed, f"{len(failed)} tiles failed; not writing partial results"

SITE_CHUNK = 20_000
site_list = geom_to_site["SITE_ID"].unique()
n_written, writer, preview = 0, None, None
with fsspec.open(RESULT_URI, "wb", **(_storage_options(RESULT_URI) or {})) as f:
    for i in range(0, len(site_list), SITE_CHUNK):
        chunk_map = geom_to_site[geom_to_site["SITE_ID"].isin(site_list[i:i + SITE_CHUNK])]
        chunk = (
            by_geom[by_geom["GEOM_ID"].isin(chunk_map["GEOM_ID"])]
            .merge(chunk_map, on="GEOM_ID")
            .groupby(["SITE_ID", *CONTEXT_NAMES], as_index=False)[VALUE_LAYERS].sum()
        )
        chunk = chunk[chunk["area_ha"] > 0]
        if chunk.empty:
            continue
        chunk = chunk.assign(
            aoi_id=chunk["SITE_ID"].astype(str),
            aoi_type="protected_area",
            natural_lands_class=chunk["natural_lands_class"].astype(np.uint8),
            alert_date=ALERT_EPOCH + pd.to_timedelta(chunk["alert_date"].astype("int64"), unit="D"),
            alert_confidence=chunk["alert_confidence"].astype("int64").map(CONFIDENCE),
        )[["aoi_id", "aoi_type", "SITE_ID", "natural_lands_class", "alert_date", "alert_confidence", "area_ha"]]
        chunk = chunk.sort_values(["aoi_id", "natural_lands_class", "alert_confidence", "alert_date"])
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        table = table.set_column(table.schema.get_field_index("alert_date"), "alert_date",
                                 table["alert_date"].cast(pa.date32()))
        if writer is None:
            writer = pq.ParquetWriter(f, table.schema)
            preview = chunk.head()
        writer.write_table(table)
        n_written += len(chunk)
    if writer is not None:
        writer.close()

print(f"Saved {n_written:,} rows for {len(site_list):,} sites to {RESULT_URI}")
preview

In [ ]:
# ---------------------------------------------------------------------------
# Spot check against the analytics API (live OTF path) for a few protected areas
# ---------------------------------------------------------------------------
# The API fetches geometries from the Data API's WDPA by site_pid (it may be a different
# WDPA release) and returns one row per date and confidence. Compare totals for the same
# date range and land filter.
import time, requests

API_URL = "https://analytics.globalnaturewatch.org/v0/land_change/integrated_alerts/analytics"
# Brazilian sites used throughout the TCL investigation, all with heavy alert activity:
# Xikrin do Rio Catete, Apyterewa, APA Triunfo do Xingu, Kayapó, APA Margem Direita
# do Rio Negro, Estação Ecológica Soldado da Borracha. The big ones (Kayapó, Triunfo)
# can take the API a few minutes each.
CHECK_SITES = [34001, 33995, 352203, 34004, 352147, 555758964]
CHECK_START, CHECK_END = "2024-01-01", "2024-12-31"
CHECK_LAND_FILTER = None  # None, "natural_lands" or "natural_forests"


def _poll_json(url):
    """GET a status link; None (plus the response) on a gateway error or non-JSON body,
    which the API returns while it's busy with a large area."""
    try:
        r = requests.get(url, timeout=120)
    except requests.RequestException as e:
        return None, e
    if r.status_code >= 500 or not r.content:
        return None, r
    try:
        return r.json(), r
    except ValueError:
        return None, r


def api_integrated_alerts(pids, timeout_s=1800, max_bad_polls=20):
    body = {"aoi": {"type": "protected_area", "ids": [str(p) for p in pids]},
            "start_date": CHECK_START, "end_date": CHECK_END, "land_filter": CHECK_LAND_FILTER}
    r = requests.post(API_URL, json=body, timeout=60)
    if not r.ok:
        raise RuntimeError(f"{r.status_code}: {r.text[:500]}")
    link, t0, bad, res = r.json()["data"]["link"], time.time(), 0, None
    while time.time() - t0 < timeout_s:
        payload, resp = _poll_json(link)
        if payload is None:
            bad += 1
            if bad > max_bad_polls:
                detail = (f"{resp.status_code}: {resp.text[:300]!r}" if hasattr(resp, "status_code") else repr(resp))
                raise RuntimeError(f"status link kept failing ({bad} polls); last: {detail}")
            time.sleep(min(60, 5 * bad))  # back off while the API is busy
            continue
        res = payload["data"]
        if res["status"] in ("saved", "failed"):
            break
        time.sleep(int(resp.headers.get("Retry-After", 5)) if resp.headers.get("Retry-After", "").isdigit() else 5)
    if res is None or res["status"] != "saved":
        raise RuntimeError(res or "timed out waiting for the API")
    return pd.DataFrame({k: v for k, v in res["result"].items() if not k.startswith("__")})


# SITE_ID -> SITE_PID, read as plain columns (no geometry): the API looks sites up by site_pid.
wdpa_attrs = pd.read_parquet(WDPA_GEOPARQUET_URI, storage_options=S3_OPTS, columns=["SITE_ID", "SITE_PID"])

ours = pd.read_parquet(RESULT_URI, storage_options=S3_OPTS, filters=[("SITE_ID", "in", CHECK_SITES)])
rows = []
for site in CHECK_SITES:
    o = ours[(ours.SITE_ID == site)
             & (pd.to_datetime(ours.alert_date) >= CHECK_START) & (pd.to_datetime(ours.alert_date) <= CHECK_END)]
    if CHECK_LAND_FILTER:
        o = o[o.natural_lands_class.isin(LAND_FILTER_CLASSES[CHECK_LAND_FILTER])]
    pids = wdpa_attrs.loc[wdpa_attrs.SITE_ID == site, "SITE_PID"].astype(str).unique()
    api = api_integrated_alerts(pids)
    rows.append({"SITE_ID": site, "ours_ha": o.area_ha.sum(), "api_ha": api.area_ha.sum()})
    print(rows[-1])
if rows:
    check = pd.DataFrame(rows).set_index("SITE_ID")
    check["rel_diff_%"] = 100 * (check.ours_ha / check.api_ha - 1)
    print(check.round(3).to_string())

## Notes

- **Alignment** follows the API and pipeline exactly: pixel area matched to the alerts grid
  within 1e-5 deg; natural lands nearest-neighbour within half a 30 m pixel, 0 outside.
- **Inclusion rule** is pixel-center, as in the API (`rio.clip` default) and the TCL notebooks.
- **Candidates** are alert pixels only (confidence 2-4, date < 5000 days), so the point test
  scales with alerts under each polygon, not with polygon or tile area.
- **Output size** grows with dates: one row per site, class, day and confidence with any
  alert. Restricting `START_DATE` / `END_DATE` shrinks both the run and the output.